# embryo-vc-bench — read-only replay of the released main tables
#
This notebook reproduces the figure/table source data from the
**already released, hash-verified artefacts**. It does NOT load
scGPT or Geneformer, does NOT recompute any preregistered score,
and does NOT re-run any W6 generator (those hashes are cited in
manuscript 3.5). Full from-scratch scoring is in ../README.md 3.9.

Run with the main environment: `./.venv/Scripts/python.exe`
or `jupyter nbconvert --execute release/reproduce.ipynb`.


In [ ]:
import csv, hashlib, json, statistics as st
from pathlib import Path
from collections import defaultdict

ROOT = Path.cwd()
if not (ROOT / 'results' / 'w5_scores.csv').exists():
    ROOT = Path.cwd().parent
RESULTS = ROOT / 'results'
RELEASE = ROOT / 'release'
FIG = RELEASE / 'figure_source_data'
print('repository root:', ROOT)

def rows(p):
    with p.open(encoding='utf-8', newline='') as f:
        return list(csv.DictReader(f))


In [ ]:
# 1) Manuscript 3.3 mean table, recomputed from the 80 released rows
scores = rows(RESULTS / 'w5_scores.csv')
g = defaultdict(list)
for r in scores: g[(r['task'], r['method'])].append(r)
tbl = []
for (task, method), rs in sorted(g.items()):
    rec = {'task': task, 'method': method, 'n': len(rs)}
    for m in ('MAE','MAE_z','DES','PDS'):
        rec[m] = round(st.mean(float(r[m]) for r in rs), 4)
    tbl.append(rec)
for r in tbl: print(r)


In [ ]:
# 2) Cross-check against the shipped source-data CSV
rel = {(r['task'], r['method']): r for r in rows(FIG / 'fig4_t2_t3_means.csv')}
bad = []
for r in tbl:
    k = (r['task'], r['method'])
    for m in ('MAE','MAE_z','DES','PDS'):
        if abs(float(rel[k][m + '_mean']) - r[m]) > 0.00006:
            bad.append((k, m))
print('cross-check:', 'OK — all 32 cells match' if not bad else bad)


In [ ]:
# 3) Direction-magnitude dissociation: paired contrasts + bootstrap CI
cc = rows(FIG / 'fig4_paired_contrasts.csv')
for r in cc:
    if r['model']=='scGPT' and r['baseline']=='no_change' and r['metric'] in ('DES','MAE'):
        print(r['task'], r['metric'], r['mean_delta'],
              'CI95', r['ci95_low'], r['ci95_high'])

# 4) Cross-platform replication (three independent held-out atlases)
for r in rows(FIG / 'fig_t2_by_heldout_atlas.csv'):
    if r['method']=='scGPT':
        print(r['held_out_atlas'], 'DES_delta', r['DES_delta_vs_no_change'],
              'MAE_delta', r['MAE_delta_vs_no_change'])


In [ ]:
# 5) Verify every SHA256 in the release manifest
man = json.loads((RELEASE / 'RELEASE_MANIFEST.json').read_text(encoding='utf-8'))
fail = []
for name, info in man['files'].items():
    p = RELEASE / name
    if not p.exists():
        fail.append((name,'MISSING')); continue
    if hashlib.sha256(p.read_bytes()).hexdigest() != info['sha256']:
        fail.append((name,'MISMATCH'))
print(len(man['files']), 'files:', 'ALL HASHES OK' if not fail else fail)
print('No model was loaded and no preregistered score was recomputed.')


In [ ]:
# 6) Optional Fig 4 panel (matplotlib only; tables above need no plotting lib)
try:
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(8,4))
    ax.bar(range(len(tbl)), [r['DES'] for r in tbl], color='#4C78A8')
    ax.set_xticks(range(len(tbl)))
    ax.set_xticklabels([r['task'].split('_')[0] + '\n' + r['method'] for r in tbl], fontsize=7)
    ax.set_ylabel('DES (framework-internal)'); fig.tight_layout()
    out = FIG / 'fig4_des_by_method.png'; fig.savefig(out, dpi=150)
    print('figure written:', out)
except ImportError:
    print('matplotlib not installed; figure skipped')
